In [ ]:
# Install libraries
!pip install -q sentence-transformers scikit-learn pandas numpy nltk

# Imports
import re
import nltk
import numpy as np
import pandas as pd

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sentence_transformers import SentenceTransformer, util
from sklearn.linear_model import LogisticRegression

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print("✅ Setup Complete")

✅ Setup Complete


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)

    tokens = word_tokenize(text)

    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    clean_tokens = [
        lemmatizer.lemmatize(w)
        for w in tokens
        if w not in stop_words and len(w) > 2
    ]

    return " ".join(clean_tokens)

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ SBERT Loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ SBERT Loaded


In [ ]:
freelancers = [
    {
        "id": 1,
        "skills": ["react", "javascript", "css"],
        "profile_text": "Experienced React developer building frontend applications",
        "experience": 5,
        "rating": 4.5,
        "completed_projects": 20,
        "success_rate": 0.85,
        "interaction": 0.9,
        "rate": 40
    },
    {
        "id": 2,
        "skills": ["python", "machine learning", "aws"],
        "profile_text": "Machine learning engineer with cloud deployment experience",
        "experience": 4,
        "rating": 4.2,
        "completed_projects": 15,
        "success_rate": 0.8,
        "interaction": 0.75,
        "rate": 60
    },
    {
        "id": 3,
        "skills": ["react", "ui/ux", "figma"],
        "profile_text": "Frontend UI/UX designer specializing in modern web apps",
        "experience": 6,
        "rating": 4.7,
        "completed_projects": 25,
        "success_rate": 0.9,
        "interaction": 0.95,
        "rate": 45
    },
    {
        "id": 4,
        "skills": ["node.js", "mongodb", "backend"],
        "profile_text": "Backend developer with Node.js and database expertise",
        "experience": 5,
        "rating": 4.3,
        "completed_projects": 18,
        "success_rate": 0.82,
        "interaction": 0.85,
        "rate": 35
    }
]


projects = [
    {
        "id": 101,
        "required_skills": ["react", "frontend"],
        "description": "Build a React frontend application",
        "budget": 500
    },
    {
        "id": 102,
        "required_skills": ["python", "machine learning"],
        "description": "Develop ML model for prediction",
        "budget": 800
    },
    {
        "id": 103,
        "required_skills": ["node.js", "backend"],
        "description": "Backend API development",
        "budget": 400
    }
]

In [ ]:
def skill_score(f_skills, p_skills):
    return len(set(f_skills).intersection(set(p_skills))) / len(p_skills)


def semantic_score(f_text, p_text):
    emb_f = model.encode(f_text, convert_to_tensor=True)
    emb_p = model.encode(p_text, convert_to_tensor=True)
    return util.cos_sim(emb_f, emb_p).item()


def context_score(f_skills):
    return len(f_skills) / 10  # simple assumption


def maturity_score(f):
    return min(f["experience"] / 10, 1.0)


def behavior_score(f):
    return (f["success_rate"] + f["interaction"]) / 2


def budget_score(f_rate, p_budget):
    return 1.0 if f_rate <= (p_budget / 10) else 0.5

In [ ]:
freelancer = freelancers[0] # Selecting the first freelancer for demonstration

features = {
    "skill_score": skill_score(freelancer["skills"], project["required_skills"]),
    "semantic_score": semantic_score(freelancer["profile_text"], project["description"]),
    "context_score": context_score(freelancer["skills"]),
    "maturity_score": maturity_score(freelancer),
    "behavior_score": behavior_score(freelancer),
    "budget_score": budget_score(freelancer["rate"], project["budget"])
}

print(features)

{'skill_score': 0.5, 'semantic_score': 0.7587878704071045, 'context_score': 0.3, 'maturity_score': 0.5, 'behavior_score': 0.875, 'budget_score': 1.0}


In [ ]:
import random

data = []

for _ in range(200):
    row = {
        "skill_score": random.uniform(0,1),
        "semantic_score": random.uniform(0,1),
        "context_score": random.uniform(0,1),
        "maturity_score": random.uniform(0,1),
        "behavior_score": random.uniform(0,1),
        "budget_score": random.uniform(0,1),
    }

    # simulated hiring logic
    score = (
        0.3*row["skill_score"] +
        0.25*row["semantic_score"] +
        0.15*row["context_score"] +
        0.15*row["maturity_score"] +
        0.1*row["behavior_score"] +
        0.05*row["budget_score"]
    )

    row["hired"] = 1 if score > 0.6 else 0
    data.append(row)

df = pd.DataFrame(data)

In [ ]:
X = df.drop("hired", axis=1)
y = df["hired"]

model_lr = LogisticRegression()
model_lr.fit(X, y)

print("✅ Model Trained")

✅ Model Trained


In [ ]:
weights = model_lr.coef_[0]
weights = np.abs(weights)
weights = weights / np.sum(weights)

feature_names = X.columns
weight_dict = dict(zip(feature_names, weights))

print("🔥 Learned Weights:")
for k,v in weight_dict.items():
    print(k, ":", round(v,3))

🔥 Learned Weights:
skill_score : 0.349
semantic_score : 0.244
context_score : 0.15
maturity_score : 0.148
behavior_score : 0.075
budget_score : 0.034


In [ ]:
final_score = 0

for key in features:
    final_score += weight_dict[key] * features[key]

print("🚀 Final Match Score:", round(final_score, 3))

🚀 Final Match Score: 0.578


In [ ]:
# ===============================
# MATCH ALL FREELANCERS TO ALL PROJECTS
# ===============================

for project in projects:

    print(f"\n==============================")
    print(f"🎯 Project {project['id']} → {project['required_skills']}")
    print(f"==============================")

    results = []

    for f in freelancers:

        # Feature extraction
        features = {
            "skill_score": skill_score(f["skills"], project["required_skills"]),
            "semantic_score": semantic_score(f["profile_text"], project["description"]),
            "context_score": context_score(f["skills"]),
            "maturity_score": maturity_score(f),
            "behavior_score": behavior_score(f),
            "budget_score": budget_score(f["rate"], project["budget"])
        }

        # ===============================
        # 🔥 TLAM MATCHING
        # ===============================
        alpha = weight_dict["skill_score"]
        beta = weight_dict["maturity_score"]
        gamma = weight_dict["context_score"]

        C = features["skill_score"] * features["semantic_score"]
        R = (features["maturity_score"] * features["behavior_score"]) ** 0.5
        A = (features["context_score"] + features["behavior_score"] + features["budget_score"]) / 3

        final_score = (C ** alpha) * (R ** beta) * (A ** gamma)

        results.append({
            "freelancer_id": f["id"],
            "score": round(final_score, 3),
            "skills": f["skills"]
        })

    # Rank freelancers for this project
    ranked = sorted(results, key=lambda x: x["score"], reverse=True)

    print("\n🔥 Recommended Freelancers:\n")
    for i, r in enumerate(ranked, 1):
        print(f"Rank {i} → Freelancer {r['freelancer_id']} | Score: {r['score']} | Skills: {r['skills']}")


🎯 Project 101 → ['react', 'frontend']

🔥 Recommended Freelancers:

Rank 1 → Freelancer 1 | Score: 0.675 | Skills: ['react', 'javascript', 'css']
Rank 2 → Freelancer 3 | Score: 0.559 | Skills: ['react', 'ui/ux', 'figma']
Rank 3 → Freelancer 2 | Score: 0.0 | Skills: ['python', 'machine learning', 'aws']
Rank 4 → Freelancer 4 | Score: 0.0 | Skills: ['node.js', 'mongodb', 'backend']

🎯 Project 102 → ['python', 'machine learning']

🔥 Recommended Freelancers:

Rank 1 → Freelancer 2 | Score: 0.534 | Skills: ['python', 'machine learning', 'aws']
Rank 2 → Freelancer 1 | Score: 0.0 | Skills: ['react', 'javascript', 'css']
Rank 3 → Freelancer 3 | Score: 0.0 | Skills: ['react', 'ui/ux', 'figma']
Rank 4 → Freelancer 4 | Score: 0.0 | Skills: ['node.js', 'mongodb', 'backend']

🎯 Project 103 → ['node.js', 'backend']

🔥 Recommended Freelancers:

Rank 1 → Freelancer 4 | Score: 0.749 | Skills: ['node.js', 'mongodb', 'backend']
Rank 2 → Freelancer 1 | Score: 0.0 | Skills: ['react', 'javascript', 'css']
R

In [ ]:
# OLD MODEL (Weighted Sum)
def old_score(features):
    return sum(weight_dict[k] * features[k] for k in features)


# NEW MODEL (TLAM)
def tlam_score(features):
    alpha = weight_dict["skill_score"]
    beta = weight_dict["maturity_score"]
    gamma = weight_dict["context_score"]

    C = features["skill_score"] * features["semantic_score"]
    R = (features["maturity_score"] * features["behavior_score"]) ** 0.5
    A = (features["context_score"] + features["behavior_score"] + features["budget_score"]) / 3

    return (C ** alpha) * (R ** beta) * (A ** gamma)


# Compare both
for f in freelancers:
    features = {
        "skill_score": skill_score(f["skills"], project["required_skills"]),
        "semantic_score": semantic_score(f["profile_text"], project["description"]),
        "context_score": context_score(f["skills"]),
        "maturity_score": maturity_score(f),
        "behavior_score": behavior_score(f),
        "budget_score": budget_score(f["rate"], project["budget"])
    }

    print(f"\nFreelancer {f['id']}")
    print("Old Score:", round(old_score(features), 3))
    print("TLAM Score:", round(tlam_score(features), 3))


Freelancer 1
Old Score: 0.343
TLAM Score: 0.0

Freelancer 2
Old Score: 0.248
TLAM Score: 0.0

Freelancer 3
Old Score: 0.341
TLAM Score: 0.0

Freelancer 4
Old Score: 0.713
TLAM Score: 0.749
